The numeric data for base ranking has dots in them to make numbers more readable but this caused the scrape code to regard them as float numbers which led to the 0s at the end getting deleted. For example 71.750 which is 71750 was stored as 71.75 whichs completely wrong. This piece of code will get rid of dots and give the 0s back to the numbers.

In [5]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('fulldata.csv')

def clean_base_ranking(value):
    # Convert the value to a string
    value = str(value)
    
    # If the value is for example 8679.0, remove the .0
    if value[-2:] == ".0":
        return value[:-2]
    
    # Add necessary zeros then remove the dots
    elif "." in value:
        index_of_dot = value.rfind(".")
        if len(value) - index_of_dot == 2:
            value += "00"
        elif len(value) - index_of_dot == 3:
            value += "0"
        return value.replace(".", "")
    else:
        return value

# Apply the cleaning function to the 'base ranking' column
df['baseRanking'] = df['baseRanking'].apply(clean_base_ranking)

# Write the cleaned data back to the CSV file
df.to_csv('fulldata_baseRankingFormatted.csv', index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\3614910257.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('fulldata.csv')


2019-2020 data which are connected with osym ids lack certain fields like region, universityType etc. Adding these according to the idÖsyms from other years.

In [6]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('fulldata_baseRankingFormatted.csv')

columns_to_fill = ["universityRegion", "universityType", "programType"]

# Create a dictionary for fast lookups
lookup_dict = {}
for _, row in df.iterrows():
    key = row['idOSYM']
    if key not in lookup_dict:
        lookup_dict[key] = {}
    for column in columns_to_fill:
        if pd.notnull(row[column]):
            lookup_dict[key][column] = row[column]

def complete_missing_fields(row):
    if pd.isnull(row[columns_to_fill]).any():
        key = row['idOSYM']
        if key in lookup_dict:
            for column in columns_to_fill:
                if pd.isnull(row[column]) and column in lookup_dict[key]:
                    row[column] = lookup_dict[key][column]
    return row

# Apply the complete_missing_fields function to each row
df = df.apply(complete_missing_fields, axis=1)

# Save the completed data to a new CSV file
df.to_csv('fulldata_baseRankingFormatted_missingFieldsFilled.csv', index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\1391598234.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('fulldata_baseRankingFormatted.csv')


First we need to fix the scraped data which has commas at wrong indexes for some reason.

In [8]:
df = pd.read_csv('./sources/2024+alldepartments(minMaxScore).csv')

columns_to_fix = ["baseScore", "topScore"]

def clean_min_max_score(row):
    for column in columns_to_fix:
        value = str(row[column]).replace(",", "")
        if len(value) > 5:
            value = value[:-5] + ',' + value[-5:]
        row[column] = value
    return row    
    
    
df = df.apply(clean_min_max_score, axis=1)

df.to_csv('./sources/2024+alldepartments(minMaxScore)_cleaned.csv', index=False)

We should also convert the academicYear and idOSYM of our data to string for matching with minMaxFile

In [12]:
df = pd.read_csv('fulldata_baseRankingFormatted_missingFieldsFilled.csv')

columns_to_format = ["academicYear", "idOSYM"]

def format_academicYear_idOSYM(row):
    for column in columns_to_format:
        row[column] = str(row[column]).replace(".0", "")
    return row

df = df.apply(format_academicYear_idOSYM, axis=1)
df.to_csv('fulldata_baseRankingFormatted_missingFieldsFilled_cleaned.csv', index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\2334102065.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('fulldata_baseRankingFormatted_missingFieldsFilled.csv')


Some baseScore and topScore fields of certain years are missing which will be filled from 2021-2024 minMaxScore scrapes i have in hand which are in resources.

In [17]:
import pandas as pd

# Read the CSV files
df = pd.read_csv('fulldata_baseRankingFormatted_missingFieldsFilled_cleaned.csv')
df_scraped = pd.read_csv('./sources/2024+alldepartments(minMaxScore)_cleaned.csv')

# Create a dictionary for fast lookups
scraped_dict = {}
for _, row in df_scraped.iterrows():
    key = (str(row['academicYear']).replace(".0", ""), str(row['idOSYM']).replace(".0", ""))
    scraped_dict[key] = {'baseScore': row['baseScore'], 'topScore': row['topScore']}

def fill_scraped_minMax_Score(row):
    if pd.isnull(row['baseScore']) or pd.isnull(row['topScore']):
        key = (str(row['academicYear']).replace(".0", ""), str(row['idOSYM']).replace(".0", ""))
        if key in scraped_dict:
            if pd.isnull(row['baseScore']):
                row['baseScore'] = scraped_dict[key]['baseScore']
            if pd.isnull(row['topScore']):
                row['topScore'] = scraped_dict[key]['topScore']
    return row

# Apply the fill_scraped_minMax_Score function to each row
df = df.apply(fill_scraped_minMax_Score, axis=1)

# Save the completed data to a new CSV file
df.to_csv('fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled.csv', index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\1382768850.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('fulldata_baseRankingFormatted_missingFieldsFilled_cleaned.csv')


Some baseScore and topScore fields have commas instead of dots. Also the problem with the baseRanking format of missing 0s exists here as well. Also since data contains both 2021 of the old groups data and our 2021 data ours which doesnt have topScore and baseScore fields after fixing the formats the baseScore and topScore of the old 2021 data will be copied to ours.

In [22]:
df = pd.read_csv('fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled.csv')

columns_to_clean = ["baseScore", "topScore"]

def clean_baseTopScore(row):
    # Convert the value to a string
    for column in columns_to_clean:
        value = str(row[column])
        if "," in value:
            value = value.replace(",", ".")
        if "." in value:
            index_of_dot = value.rfind(".")
            if len(value) - index_of_dot == 2:
                value += "0000"
            elif len(value) - index_of_dot == 3:
                value += "000"
            elif len(value) - index_of_dot == 4:
                value += "00"
            elif len(value) - index_of_dot == 5:
                value += "0"
        row[column] = value
    return row

def fill_2021_baseTopScore(row):
    for column in columns_to_clean:
        if row["academicYear"] == 2021 and ( pd.isnull(row[column]) or str(row[column]) == "0"  or str(row[column]) == "0.0" or str(row[column]) == "nan" ):
            key = (row['academicYear'], row['idOSYM'])
            if key in dict:
                row[column] = dict[key][column]
       
        
    return row


df = df.apply(clean_baseTopScore, axis=1)

dict = {}
for _, row in df.iterrows():
    if str(row["academicYear"]).replace(".0","") == "2021" and not pd.isnull(row["baseScore"]) and not pd.isnull(row["topScore"]):
        key = (row['academicYear'], row['idOSYM'])
        dict[key] = {'baseScore': row['baseScore'], 'topScore': row['topScore']}

df = df.apply(fill_2021_baseTopScore, axis=1)

df.to_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed.csv", index=False)


C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\2058914191.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled.csv')


Some outOfCityStudentRate, sameRegionStudentRate, top1PreferenceRatio, top3PreferenceRatio, top9PreferenceRatio columns have % xx,x type of data instead of float out of 10 with 2 accuracy digits representation. 

Plus avgOrderofPreference, avgAdmittedStudentPrefOrder fields have comma instead of dot

In [23]:
df = pd.read_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed.csv")

columns_percentage = ["outOfCityStudentRate", "sameRegionStudentRate", "top1PreferenceRatio", "top3PreferenceRatio", "top9PreferenceRatio"]

columns_comma = ["avgOrderofPreference", "avgAdmittedStudentPrefOrder"]

def number_formatter(row):
    for column in columns_percentage:
        value = str(row[column])
        if "%" in value:
            value = value.replace("% ", "")
            value = value.replace(",", ".")
            value = float(value)/10.0
            row[column] = value
            
    for column in columns_comma:
        value = str(row[column])
        if "," in value:
            value = value.replace(",", ".")
            row[column] = value
        
    return row

df = df.apply(number_formatter, axis=1)

df.to_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed_percentageCommasFixed.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\4206311434.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed.csv")


topRanking,avgAdmissionRanking(TYT),baseAdmissionRanking(TYT),totalPreference have the same issue with baseRanking. This piece of code fixes that.


In [25]:
df = pd.read_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed_percentageCommasFixed.csv")

columns_to_reformat = ["topRanking","avgAdmissionRanking(TYT)","baseAdmissionRanking(TYT)","totalPreference"
                       ,"admittedGovPref","admittedPrivPref","admittedTotalPref","admittedTotalDepartmentPref","currentStudentCount"]

def fix_dot_issue(row):
    for column in columns_to_reformat:
        value = str(row[column])
        if "." in value:
            if value[-2:] == ".0":
                value = value[:-2]
                row[column] = value
            else:
                index_of_dot = value.rfind(".")
                if len(value) - index_of_dot == 2:
                    value += "00"
                elif len(value) - index_of_dot == 3:
                    value += "0"
                value = value.replace(".","")
                row[column] = value
                
    return row


df = df.apply(fix_dot_issue, axis=1)

df.to_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed_percentageCommasFixed_dotsFixed.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\2293309223.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed_percentageCommasFixed.csv")


Tranforming top1AdmittedRatio,top3AdmittedRatio,top10AdmittedRatio from percentage to float in range of 10.

In [26]:
df = pd.read_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed_percentageCommasFixed_dotsFixed.csv")

columns_dot_from_percentage = ["top1AdmittedRatio","top3AdmittedRatio","top10AdmittedRatio"]

def percantage_fixer(row):
    for column in columns_dot_from_percentage:
        value = str(row[column])
        if "%" in value:
            value = value.replace("%", "")
            if value[-1] == ",":
                value = value.replace(",",".0")
            else:
                value.replace(",",".")
            value = float(value)/10.0
            row[column] = value
    
    return row

df = df.apply(percantage_fixer, axis=1)

df.to_csv("fulldata_clean.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_3272\1779286539.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_baseRankingFormatted_missingFieldsFilled_cleaned_missingBaseTopScoreFilled_scoresFixed_percentageCommasFixed_dotsFixed.csv")


ValueError: could not convert string to float: '22,2'